# Module 3: Python for Data Engineer

**EJ** · Stage 3 · Nov 19–24, 2026 · image: python-training:3.0

## Section 3.1: Datetime

### B-3.1.1

Create 30 timestamp strings in three formats, parse each group with an explicit format, and count the NaTs produced by a deliberately bad value (Lab 3.1 step 1).

In [1]:
# Will use the existing timestamps.csv file

import pandas as pd

df = pd.read_csv("../data/module3/timestamps.csv")

print(df.head())
print("Total rows:", len(df))

   event_id source_system               ts_raw
0         1           crm     2026-03-01 08:00
1         2           erp           02/03/2026
2         3           web  Mar 3, 2026 8:00 AM
3         4           crm     2026-03-03 11:00
4         5           erp           04/03/2026
Total rows: 30


The file has three columns: **event_id, source_system, and ts_raw** and 30 rows.

There are three source system groups: **crm, erp, and web**

In [2]:
# The explicit formats for each group

formats = {
    "crm": "%Y-%m-%d %H:%M",
    "erp": "%d/%m/%Y",
    "web": "%b %d, %Y %I:%M %p",
}

In [3]:
df["ts_parsed"] = pd.NaT # Start with an empty timestamp for every row

# Stop if a source_system has no format, so its rows aren't miscounted as bad
assert df["source_system"].isin(formats).all(), "Unknown source_system found"

# Parsing each group with the format
for source_system, ts_format in formats.items():
    rows = df["source_system"] == source_system

    df.loc[rows, "ts_parsed"] = pd.to_datetime(df.loc[rows, "ts_raw"], format=ts_format, errors="coerce")

In [4]:
df

,event_id,source_system,ts_raw,ts_parsed
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00
1,2,erp,02/03/2026,2026-03-02 00:00:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00
4,5,erp,04/03/2026,2026-03-04 00:00:00
5,6,web,"Mar 5, 2026 11:00 AM",2026-03-05 11:00:00
6,7,crm,2026-03-05 14:00,2026-03-05 14:00:00
7,8,erp,31/02/2026,NaT
8,9,web,"Mar 7, 2026 2:00 PM",2026-03-07 14:00:00
9,10,crm,2026-03-07 17:00,2026-03-07 17:00:00


In [5]:
bad = df[df["ts_parsed"].isna()] # returns only the rows where ts_parsed is NaT
bad

,event_id,source_system,ts_raw,ts_parsed
7,8,erp,31/02/2026,NaT
20,21,web,"Marz 9, 2026 3:00 PM",NaT


Among the 30 rows, only two are not valid timestamps:

- event_id 8 - **31/02/2026**: there is no february 31
- event_id 21 - **Marz 9, 2026** 3:00 PM: Marz is not a valid month name

In [6]:
assert len(bad) == 2, f"Expected 2 bad timestamps, got {len(bad)}"

### B-3.1.2

Make the datetimes timezone-aware, convert to UTC, and show both side by side (Lab 3.1 step 2).

In [7]:
# Assigning a local timezone
df["ts_local"] = df["ts_parsed"].dt.tz_localize("Asia/Manila")
df.head()

,event_id,source_system,ts_raw,ts_parsed,ts_local
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00


In [8]:
# Converting the local UTC+8 to UTC+0
df["ts_utc"] = df["ts_local"].dt.tz_convert("UTC")
df.head()

,event_id,source_system,ts_raw,ts_parsed,ts_local,ts_utc
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00


In [9]:
# The two timezone's side-by-side

df[["event_id", "ts_local", "ts_utc"]].head()

,event_id,ts_local,ts_utc
0,1,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00


For event_id 2, **March 2 at 00:00** in Manila or 12 AM becomes **March 1 at 16:00** or 4PM in UTC.

Because Manila is **UTC+8**, any local time before 8:00 AM falls on the previous day in UTC

### B-3.1.3

Derive year, month, weekday, and hour columns, and resample to daily counts (Lab 3.1 step 3).

In [10]:
# Using the same ts_local so it follows local time

df["year"] = df["ts_local"].dt.year
df["month"] = df["ts_local"].dt.month
df["weekday"] = df["ts_local"].dt.day_name()
df["hour"] = df["ts_local"].dt.hour

df[["event_id", "ts_local", "year", "month", "weekday", "hour"]].head() 

,event_id,ts_local,year,month,weekday,hour
0,1,2026-03-01 08:00:00+08:00,2026.0,3.0,Sunday,8.0
1,2,2026-03-02 00:00:00+08:00,2026.0,3.0,Monday,0.0
2,3,2026-03-03 08:00:00+08:00,2026.0,3.0,Tuesday,8.0
3,4,2026-03-03 11:00:00+08:00,2026.0,3.0,Tuesday,11.0
4,5,2026-03-04 00:00:00+08:00,2026.0,3.0,Wednesday,0.0


Since there are two NaT values, the output shows float versions for the year, month, and hour. Missing values are stored as NaN, which is a float, so pandas turns the whole column into float to fit them.

In [11]:
# Resampling to daily counts

daily_counts = (df.resample("D", on="ts_local").size().rename("event_count"))
# D groups events into calendar days using ts_local

print("Days:", len(daily_counts))
daily_counts

Days: 22


ts_local
2026-03-01 00:00:00+08:00    1
2026-03-02 00:00:00+08:00    1
2026-03-03 00:00:00+08:00    2
2026-03-04 00:00:00+08:00    1
2026-03-05 00:00:00+08:00    2
2026-03-06 00:00:00+08:00    0
2026-03-07 00:00:00+08:00    2
2026-03-08 00:00:00+08:00    1
2026-03-09 00:00:00+08:00    2
2026-03-10 00:00:00+08:00    1
2026-03-11 00:00:00+08:00    2
2026-03-12 00:00:00+08:00    1
2026-03-13 00:00:00+08:00    1
2026-03-14 00:00:00+08:00    1
2026-03-15 00:00:00+08:00    1
2026-03-16 00:00:00+08:00    1
2026-03-17 00:00:00+08:00    1
2026-03-18 00:00:00+08:00    2
2026-03-19 00:00:00+08:00    1
2026-03-20 00:00:00+08:00    2
2026-03-21 00:00:00+08:00    1
2026-03-22 00:00:00+08:00    1
Name: event_count, dtype: int64

There are **28 valid events over 22 days** (the 2 invalid timestamps are excluded). Mar 6 shows 0 because resample lists every calendar day in the range, even days with no events.

In [12]:
valid_count = df["ts_local"].notna().sum()
assert daily_counts.sum() == valid_count, "Daily counts don't add up to the valid events"

### B-3.1.4

Compute the time between two events with `timedelta` (Lab 3.1 step 4).

In [13]:
# Will use the evets.csv file for this, specifically the ordered_at and shipped_at columns

events = pd.read_csv("../data/module3/events.csv")
iso_format = "%Y-%m-%dT%H:%M:%S" # original format of the timestamp

# Parse the format into the ordered_at and shipped_at events
for column in ["ordered_at", "shipped_at"]:
    events[column] = pd.to_datetime(events[column], format=iso_format, errors="coerce")

print(events.shape)
events.head()

(10, 3)


,event_id,ordered_at,shipped_at
0,E001,2026-03-02 15:45:00,2026-03-03 18:45:00
1,E002,2026-03-03 10:45:00,2026-03-06 09:45:00
2,E003,2026-03-04 16:15:00,2026-03-06 09:15:00
3,E004,2026-03-05 13:45:00,2026-03-10 01:45:00
4,E005,2026-03-06 15:45:00,NaT


The file has **three columns and 10 rows**. One row has a **missing** shipped_at value.

In [14]:
# timedelta handles a missing value (event 5)
events["order_to_ship_duration"] = events["shipped_at"] - events["ordered_at"]

events

,event_id,ordered_at,shipped_at,order_to_ship_duration
0,E001,2026-03-02 15:45:00,2026-03-03 18:45:00,1 days 03:00:00
1,E002,2026-03-03 10:45:00,2026-03-06 09:45:00,2 days 23:00:00
2,E003,2026-03-04 16:15:00,2026-03-06 09:15:00,1 days 17:00:00
3,E004,2026-03-05 13:45:00,2026-03-10 01:45:00,4 days 12:00:00
4,E005,2026-03-06 15:45:00,NaT,NaT
5,E006,2026-03-07 15:15:00,2026-03-09 04:15:00,1 days 13:00:00
6,E007,2026-03-08 08:00:00,2026-03-11 04:00:00,2 days 20:00:00
7,E008,2026-03-09 10:15:00,2026-03-09 19:15:00,0 days 09:00:00
8,E009,2026-03-10 17:15:00,2026-03-11 00:15:00,0 days 07:00:00
9,E010,2026-03-11 17:00:00,2026-03-15 14:00:00,3 days 21:00:00


In [15]:
print("Average:", events["order_to_ship_duration"].mean())
print("Longest:", events["order_to_ship_duration"].max())

Average: 2 days 03:13:20
Longest: 4 days 12:00:00


E005 has no **shipped_at**, so it parses to NaT. Any subtraction with NaT gives NaT, so its order_to_ship_duration is NaT too. 

**.mean() and .max() skip NaT**, which means the average covers 9 orders, not 10.

**Why silent parsing failures are dangerous and how to make them loud?**

Silent parsing are dangerious because a report can look complete even when it leaves out records. In `timestamps.csv`, `errors="coerce"` turned two invalid timestamps into `NaT` and the script kept running, but he daily counts included only 28 out of 30 events.


I could've just used `errors="raise"` to stop when pandas finds an invalid timestamp and make the failures loud. Instead, I used `errors="coerce"` and made it loud by counting **NaTs**, printing the failed rows, and asserting the count matched the bad inputs I'd found.

## Section 3.2: JSON, CSV, and Excel

### 3.2 Design choices

**Data:** the provided `orders_nested.json` (12 orders, nested customer and items) and `large_events.csv` (50,000 rows), from the module's data folder.

**Package:** openpyxl 3.1.5, added to requirements.txt for the Excel round trip.

**One row = one item** from an order's `items` list. Each row also carries the order's `order_id` and `ordered_at`, plus the customer's `id` and `name`. Email and city are left out because this lab doesn't need them.

**Prediction:** 36 rows, the total number of items across the 12 orders (counted in code below).

**Chunked read**: large_events.csv (provided, 50,000 rows) is read in chunks of 10,000 rows, so the loop should run 5 times. In a real pipeline, chunking lets you process files too big to fit in memory, because only one chunk is loaded at a time. 10,000 keeps each chunk small while needing only a few loops.

### B-3.2.1

Create a small nested JSON file, flatten it to one row per item, and save it to CSV and Excel (Lab 3.2 step 1).

In [16]:
# Will use the existing orders_nested.json file

import json
import pandas as pd

with open("../data/module3/orders_nested.json", encoding="utf-8") as file:
    orders = json.load(file) # reads the file into a python list

In [17]:
# Since each item should be its own row, there should be 36 rows in total (prediction)

expected_rows = sum(len(order["items"]) for order in orders)
expected_rows

36

In [18]:
print("Orders:", len(orders))
print(json.dumps(orders[0], indent=2)) # one order sample

Orders: 12
{
  "order_id": "A-1001",
  "ordered_at": "2026-04-01",
  "customer": {
    "id": "C-004",
    "name": "Customer A",
    "email": "customera@example.test",
    "city": "Davao City"
  },
  "items": [
    {
      "sku": "P-102",
      "name": "Mouse",
      "qty": 1,
      "unit_price": 14.0
    },
    {
      "sku": "P-105",
      "name": "USB Cable",
      "qty": 1,
      "unit_price": 6.5
    },
    {
      "sku": "P-104",
      "name": "Desk Lamp",
      "qty": 1,
      "unit_price": 18.0
    },
    {
      "sku": "P-103",
      "name": "Keyboard",
      "qty": 2,
      "unit_price": 25.0
    }
  ]
}


In [19]:
# Flattening the items list, one row per item

items = pd.json_normalize(orders, record_path="items",
    meta=["order_id", "ordered_at", ["customer", "id"], ["customer", "name"], ["customer", "email"], ["customer", "city"]],
    sep="_",
)

items.head()

,sku,name,qty,unit_price,order_id,ordered_at,customer_id,customer_name,customer_email,customer_city
0,P-102,Mouse,1,14.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
1,P-105,USB Cable,1,6.5,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
2,P-104,Desk Lamp,1,18.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
3,P-103,Keyboard,2,25.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City
4,P-103,Keyboard,4,25.0,A-1002,2026-04-02,C-001,Customer B,customerb@example.test,Baguio


In [20]:
# Checking the quality of the flattened rows

quality = []
for col in items.columns:
    quality.append({
        "column": col,
        "dtype": str(items[col].dtype),
        "nulls": int(items[col].isna().sum()),
        "unique": items[col].nunique(),
    })

pd.DataFrame(quality)

,column,dtype,nulls,unique
0,sku,object,0,6
1,name,object,0,6
2,qty,int64,0,5
3,unit_price,float64,0,6
4,order_id,object,0,12
5,ordered_at,object,0,12
6,customer_id,object,0,8
7,customer_name,object,0,12
8,customer_email,object,7,10
9,customer_city,object,0,4


There are 7 rows which have a null `customer_email`. The code below checks the original json file which specific orders are these. 

In [21]:
[order for order in orders if order["customer"]["email"] is None]

[{'order_id': 'A-1004',
  'ordered_at': '2026-04-04',
  'customer': {'id': 'C-001',
   'name': 'Customer D',
   'email': None,
   'city': 'Davao City'},
  'items': [{'sku': 'P-103', 'name': 'Keyboard', 'qty': 1, 'unit_price': 25.0},
   {'sku': 'P-104', 'name': 'Desk Lamp', 'qty': 1, 'unit_price': 18.0},
   {'sku': 'P-105', 'name': 'USB Cable', 'qty': 2, 'unit_price': 6.5}]},
 {'order_id': 'A-1010',
  'ordered_at': '2026-04-10',
  'customer': {'id': 'C-006',
   'name': 'Customer J',
   'email': None,
   'city': 'Quezon City'},
  'items': [{'sku': 'P-100', 'name': 'Notebook', 'qty': 5, 'unit_price': 4.5},
   {'sku': 'P-104', 'name': 'Desk Lamp', 'qty': 3, 'unit_price': 18.0},
   {'sku': 'P-103', 'name': 'Keyboard', 'qty': 1, 'unit_price': 25.0},
   {'sku': 'P-101', 'name': 'Pen', 'qty': 4, 'unit_price': 1.2}]}]

Seems like `A-1004` contains 3 items and `A-1010` contains 4 items, resulting into the 7 null emails in our checking. `json_normalize` copies each order-level field onto every item row, so one missing email becomes one null per item in that order.

In [22]:
assert len(items) == expected_rows, "Number of flattened rows do not match the expected number of rows"

# Saving the result into CSV and Excel

items.to_csv("../data/module3/orders_nested_flattened.csv", index=False)
items.to_excel("../data/module3/orders_nested_flattened.xlsx", index=False)

### B-3.2.2

Read the CSV back with explicit `dtype` and `usecols`, read the Excel file back, and compare the three versions (Lab 3.2 step 2).

In [23]:
# Reading the CSV with explicit dtype and usecols

column_types = {
    "order_id": "string",
    "sku": "string",
    "qty": "int64",
    "unit_price": "float64",
    "customer_email": "string",
}

columns = list(column_types)

csv_items = pd.read_csv("../data/module3/orders_nested_flattened.csv", usecols=columns, dtype=column_types)
excel_items = pd.read_excel("../data/module3/orders_nested_flattened.xlsx", usecols=columns)

In [24]:
# Comparing column types between the three versions

pd.DataFrame({
    "Original": items[columns].dtypes,
    "CSV": csv_items[columns].dtypes,
    "Excel": excel_items[columns].dtypes,
})

,Original,CSV,Excel
order_id,object,string[python],object
sku,object,string[python],object
qty,int64,int64,int64
unit_price,float64,float64,float64
customer_email,object,string[python],object


The datatypes between the original normalized json file and the excel file remain exactly the same. For the CSV file, since I explicitly set the dtype, `order_id`, `sku`, and `customer_email` shows `string[python]` instead.

In [25]:
# Comparing if the row count and number of missing values stayed the same

tables = {"Original": items, "CSV": csv_items, "Excel": excel_items}

missing_emails = items[items["customer_email"].isna()]
first_missing_row = missing_emails.index[0]

summary = {}
for label, table in tables.items():
    summary[label] = {"Rows": len(table)}

    for column in csv_items.columns:
        summary[label][f"Missing {column}"] = table[column].isna().sum()

    email = table.loc[first_missing_row, "customer_email"]
    summary[label]["Missing email dtype"] = repr(email)

pd.DataFrame(summary)

,Original,CSV,Excel
Rows,36,36,36
Missing sku,0,0,0
Missing qty,0,0,0
Missing unit_price,0,0,0
Missing order_id,0,0,0
Missing customer_email,7,7,7
Missing email dtype,None,<NA>,nan


In [26]:
# Checking if the resuls match

original_typed = items[csv_items.columns].astype(column_types)
assert csv_items.equals(original_typed), "CSV round trip changed the data"
assert excel_items.equals(items[excel_items.columns]), "Excel round trip changed the data"
print("CSV and Excel both match the original")

CSV and Excel both match the original


**What changes when data makes a round trip between formats?**

The row and missing count for each three versions matched, including the 7 missing emails. What changed was the **CSV's text columns** and how pandas displayed a missing email. For a missing email, `json.load()` converted JSON `null` to Python `None`. Pandas read the empty CSV field as `<NA>` and the blank Excel cell as `NaN`.

### B-3.2.3

Read a larger CSV in chunks and compute a running total (Lab 3.2 step 3).

In [27]:
# Will use the provided large_events file which contains 50K events

chunks = pd.read_csv("../data/module3/large_events.csv", usecols=["amount"], chunksize=10_000)

running_total = 0 # Will hold the total of amounts

for chunk_number, chunk in enumerate(chunks, start=1):
    running_total += chunk["amount"].sum()
    print(f"After chunk {chunk_number}: {running_total:,.2f}")

After chunk 1: 2,519,323.26
After chunk 2: 5,046,021.47
After chunk 3: 7,551,027.03
After chunk 4: 10,048,405.97
After chunk 5: 12,542,076.33


`usecols=["amount"]` loaded only the column I needed, so each chunk held 1 of the 3 columns. Chunking limits how many rows are in memory, and usecols limits how many columns.

In [28]:
full_total = pd.read_csv("../data/module3/large_events.csv")["amount"].sum()

assert chunk_number == 5, f"Expected 5 chunks, got {chunk_number}"
assert round(running_total, 2) == round(full_total, 2), "Chunked total doesn't match the full read"
print(f"Chunks: {chunk_number} | Chunked total: {running_total:,.2f} | Full read: {full_total:,.2f}")

Chunks: 5 | Chunked total: 12,542,076.33 | Full read: 12,542,076.33


## Section 3.3: Avro

### 3.3 Design choices

**1. Data.** I'll reload the 36 item rows from `orders_nested_flattened.csv`, so 3.3 runs without rerunning the 3.2 cells. I'll write them to `orders.avro`, read them back into a DataFrame, and check that all 36 rows round-trip.

**2. Optional field.** `customer_email` uses `["null", "string"]` because 7 of the 36 rows have no email. Missing values from the CSV (NaN) are converted to Python `None` before writing, since Avro's null is `None`.

**3. Field types.**

| Avro type | Fields |
| --- | --- |
| `string` | `sku`, `name`, `order_id`, `ordered_at`, `customer_id`, `customer_name`, `customer_city` |
| `int` | `qty` |
| `double` | `unit_price` |
| `["null", "string"]` | `customer_email` |

**4. Size comparison.** I'll also save the same 36 rows as `orders_nested_flattened.json` (an array of records, one per item) and compare file sizes in bytes for the CSV, JSON and Avro versions, with the same fields and row order in all three.

**5. Schema evolution.** I'll add a `string` field `order_status` with default `"unknown"`, since the source has no status. Reading the original `orders.avro` with the new reader schema should give all 36 rows `"unknown"` without rewriting the file.

**Package:** fastavro 1.13.1, added to `requirements.txt`.

### B-3.3.1

Define an Avro schema for the flattened order-item table with one optional field (Lab 3.3 step 1).

In [29]:
import pandas as pd
from fastavro import parse_schema, reader, writer

items_df = pd.read_csv("../data/module3/orders_nested_flattened.csv")

# Avro schema
schema = {
    "type": "record",
    "name": "OrderItem",
    "fields": [
        {"name": "sku", "type": "string"},
        {"name": "name", "type": "string"},
        {"name": "qty", "type": "int"},
        {"name": "unit_price", "type": "double"},
        {"name": "order_id", "type": "string"},
        {"name": "ordered_at", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "customer_name", "type": "string"},
        {"name": "customer_email", "type": ["null", "string"], "default": None}, # optional field
        {"name": "customer_city", "type": "string"},
    ],
}

# Raises an error if the schema is invalid before any data is written
parsed_schema = parse_schema(schema)
print("Schema is valid.")

Schema is valid.


### B-3.3.2

Write `orders.avro`, read it back, and load it into a DataFrame (Lab 3.3 step 2).

In [30]:
records = items_df.astype(object).where(items_df.notna(), None).to_dict(orient="records")

# wb (write binary) - creates the file (bytes) if needed or clears if already exists
with open("../data/module3/orders.avro", "wb") as file:
    writer(file, parsed_schema, records)

print("Wrote", len(records), "records to orders.avro")

Wrote 36 records to orders.avro


Without the `.where(items_df.notna(), None)` where the conversion from NaN to None happens, an error appeared when I first wrote the file without it: `ValueError: nan (type <class 'float'>) do not match ['null', 'string'] on field customer_email` since pandas represents a missing email as NaN which is a float and the schema's `customer_email` field only accepts `null (None)` or string.

In [31]:
# Read the avro file back and load it into a dataframe.

with open("../data/module3/orders.avro", "rb") as file:
    items_from_avro = pd.DataFrame(reader(file))

In [32]:
print("Rows read back:", len(items_from_avro))
assert len(items_from_avro) == len(items_df) == 36
assert items_from_avro.equals(items_df), "Avro round trip changed the data"
print("Avro round trip matches the original")

Rows read back: 36
Avro round trip matches the original


### B-3.3.3

Compare file sizes of CSV, JSON, and Avro for the same data in a table (Lab 3.3 step 3).

In [33]:
from pathlib import Path

data_dir = Path("../data/module3")
file_paths = {
    "CSV": data_dir / "orders_nested_flattened.csv",
    "JSON": data_dir / "orders_nested_flattened.json",
    "Avro": data_dir / "orders.avro",
    "Avro (deflate)": data_dir / "orders_deflate.avro",
}

#  Convert the items_df (CSV) into JSON
items_df.to_json(file_paths["JSON"], orient="records")

# codec="deflate" here means that compression is enabled for the avro file
with file_paths["Avro (deflate)"].open("wb") as file:
    writer(file, parsed_schema, records, codec="deflate")

file_sizes = pd.DataFrame([
    {"format": label, "file": path.name, "bytes": path.stat().st_size}
    for label, path in file_paths.items()
])

file_sizes

,format,file,bytes
0,CSV,orders_nested_flattened.csv,3155
1,JSON,orders_nested_flattened.json,8003
2,Avro,orders.avro,3729
3,Avro (deflate),orders_deflate.avro,1030


The `uncompressed Avro file (3729)` is bigger than the `CSV (3155)` since Avro stores the full schema in its header; the deflated version shows how much space compression saves on the same Avro records `(1030)`. With only 36 rows, the header is a big share of the file but on a larger data it barely counts. `JSON` was the largest because it repeats all 10 field names for every item.

### B-3.3.4

Add a field to the schema and show reading old data with the new schema (Lab 3.3 step 4).

In [34]:
from copy import deepcopy

reader_schema = deepcopy(schema)
reader_schema["fields"].append(
    {"name": "order_status", "type": "string", "default": "unknown"}
) # Adding a new order_status field to the schema, with a default value of unknown

In [35]:
# loading the old data with the new schema

with file_paths["Avro"].open("rb") as file:
    items_with_status = pd.DataFrame(
        reader(file, reader_schema=parse_schema(reader_schema))
    )
    
assert "order_status" in items_with_status.columns, "Missing order_status column"
assert len(items_with_status) == 36, f"Expected 36 rows, got {len(items_with_status)}"
assert items_with_status["order_status"].eq("unknown").all(), "Expected 'unknown' for every item"

items_with_status["order_status"].value_counts()
items_with_status.head()

,sku,name,qty,unit_price,order_id,ordered_at,customer_id,customer_name,customer_email,customer_city,order_status
0,P-102,Mouse,1,14.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City,unknown
1,P-105,USB Cable,1,6.5,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City,unknown
2,P-104,Desk Lamp,1,18.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City,unknown
3,P-103,Keyboard,2,25.0,A-1001,2026-04-01,C-004,Customer A,customera@example.test,Davao City,unknown
4,P-103,Keyboard,4,25.0,A-1002,2026-04-02,C-001,Customer B,customerb@example.test,Baguio,unknown


Even though the old data didn’t have an `order_status` field, we could read it using the new schema because Avro supplied the `default value "unknown"` for each row.

**Why a schema-carrying format matters for pipelines?**

A schema-carrying format like Avro tells each pipeline stage how to read the data. The file includes the field names and types, so a downstream job knows that `qty` is an `integer` and `customer_email` can be `null.` This also helps with schema changes like in our lab where the original file didn’t have an `order_status` field. When we read it with the new schema, Avro supplied the `default value "unknown"` for all 36 rows without us having to rewrite the file.

Avro also checks the data against the schema when writing. When I tried to write a `NaN` email, `fastavro` **rejected** it straight away, while CSV would have stored an `empty cell` and left the reader to guess what it meant, as in 3.2's round trip.

## Section 3.4: Web scraping (local sandbox)

### B-3.4.1

Add the `sandbox-site` service under the `scrape` profile and start it (Lab 3.4 step 1).

### 3.4 Design choices

**Target.** The generated local sandbox site, because the mentor's site files weren't in the repo (`sandbox_site/`, built by `scripts/build_sandbox_site.py`). From Jupyter I start at `http://sandbox-site/page-1.html` and follow the "Next" link until the last listing page.

**User agent.** Every request sends:
`AndreaTrainingScraper/1.0 (Python Training Lab 3.4; contact: Andrea via the training repo)`

**Rules and delay.** Before fetching any listing, I read `http://sandbox-site/robots.txt` and record:
- the date I checked it
- which paths my user agent may visit
- its `Crawl-delay`, in seconds

I wait that long between requests and check every URL with robots.txt before fetching it. To test the rule, the scraper skips `/admin/` without requesting it.

**Fields per product.** ID, name, category, price, rating, image URL, and product URL. Prices are converted to float, relative URLs become full URLs, and each product is one row in `data/module3/products.csv`.

**Missing ratings.** Some products have "No reviews yet" instead of a `data-rating`, so they get stored as `None`. 

**Images and packages.** `nginx:1.28-alpine` serves the site; `requests==2.34.2` fetches pages; `beautifulsoup4==4.15.0` parses the HTML.

### B-3.4.2

Fetch and parse the site's `robots.txt`, and write down what it allows for the trainee's user agent and the date checked (Lab 3.4 step 2).

**What's the difference between "allowed by robots.txt" and "authorized**

"Allowed by robots.txt" means a page is within the site's crawling rules for my scraper. "Authorized" means I have permission to collect the data for my specific purpose; this can come from the site's terms and conditions or directly from its owner.

In [36]:
# Setting up the scraper

import requests
from urllib import robotparser

BASE_URL = "http://sandbox-site"
USER_AGENT = "AndreaTrainingScraper/1.0 (Python Training Lab 3.4; contact: Andrea via the training repo)"
HEADERS = {"User-Agent": USER_AGENT}

In [37]:
# Fetching robots.txt

response = requests.get(BASE_URL + "/robots.txt", headers=HEADERS, timeout=10)
response.raise_for_status() # raises an error if the server reports a problem
print(response.text)

# robots.txt for the NOIRE practice shop
User-agent: *
Disallow: /admin/
Crawl-delay: 1



In [38]:
# Interpret and check the rules in the robots.txt

robots = robotparser.RobotFileParser()
robots.parse(response.text.splitlines())

crawl_delay = robots.crawl_delay(USER_AGENT)
DELAY = crawl_delay if crawl_delay is not None else 1 # If there's no crawl_delay, use 1 second
print("Crawl delay (seconds):", crawl_delay)

# Which pages can the user agent fetch
paths = [f"/page-{n}.html" for n in range(1, 6)] + ["/admin/"] # Page 1-6 and /admin

for path in paths:
    allowed = robots.can_fetch(USER_AGENT, BASE_URL + path)
    print(path, "allowed:", allowed)

Crawl delay (seconds): 1
/page-1.html allowed: True
/page-2.html allowed: True
/page-3.html allowed: True
/page-4.html allowed: True
/page-5.html allowed: True
/admin/ allowed: False


`robots.txt` checked on 2026-11-24: allows all pages except `/admin/`, crawl delay 1s.

### B-3.4.3

Scrape one listing page, extracting at least 4 fields per item (Lab 3.4 step 3).

In [39]:
# Parse the HTML

from time import sleep
from bs4 import BeautifulSoup

# Scraping the first page
page_url = BASE_URL + "/page-1.html"

if not robots.can_fetch(USER_AGENT, page_url):
    raise ValueError("robots.txt disallows page-1.html")

sleep(DELAY) # Pause for 1 second
response = requests.get(page_url, headers=HEADERS, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")
print(soup.prettify()[640:1300])

<section class="product-grid">
    <article class="product" data-product-id="NR-001">
     <a class="product-link" href="product/steel-track-jacket.html">
      <div class="product-image" style="--tone: 52%">
       <img alt="Steel Track Jacket" src="img/jacket.svg"/>
      </div>
      <h2 class="product-name">
       Steel Track Jacket
      </h2>
     </a>
     <p class="product-category">
      Outerwear
     </p>
     <p class="price-line">
      <span class="price">
       ₱4,199.00
      </span>
     </p>
     <p class="rating" data-rating="3.4">
      ★ 3.4
      <span class="review-count">
       (382)
      </span>
     </p>
    </article>
  


A product is contained within an `article tag` and has a `product` class. It also contains the following: `product-id`, `name`, `category`, `price`, `rating`, `image_url`, and `product_url`.

**Why requests and Beatiful Soup works here?**

`requests` and Beautiful Soup work here because the sandbox sends the product listings as static HTML. When `requests.get()` fetches page 1, the response already contains the product cards, prices, and Next link. Beautiful Soup can then find each `article.product` and extract the URL for the next page.

In [40]:
# Extracting fields from the item

from urllib.parse import urljoin

def parse_products(soup, page_url):
    products = []

    for card in soup.select("article.product"):
        rating_tag = card.select_one(".rating[data-rating]")

        products.append({
            "product_id": card["data-product-id"],
            "name": card.select_one(".product-name").get_text(strip=True),
            "category": card.select_one(".product-category").get_text(strip=True),
            "price": card.select_one(".price").get_text(strip=True),
            "rating": rating_tag["data-rating"] if rating_tag else None,
            "image_url": urljoin(page_url, card.select_one("img")["src"]),
            "product_url": urljoin(page_url, card.select_one("a.product-link")["href"]),
        })

    return products

products = parse_products(soup, "/page-1.html")

print(f"Found {len(products)} products on page 1.")
pd.DataFrame(products)

Found 12 products on page 1.


,product_id,name,category,price,rating,image_url,product_url
0,NR-001,Steel Track Jacket,Outerwear,"₱4,199.00",3.4,/img/jacket.svg,/product/steel-track-jacket.html
1,NR-002,Raw Coach Jacket,Outerwear,"₱1,889.30",4.1,/img/jacket.svg,/product/raw-coach-jacket.html
2,NR-003,Raw Knit Polo,Tops,"₱1,599.00",4.8,/img/tee.svg,/product/raw-knit-polo.html
3,NR-004,Eclipse Hoodie,Tops,"₱1,699.00",3.8,/img/hoodie.svg,/product/eclipse-hoodie.html
4,NR-005,Eclipse Shell Jacket,Outerwear,"₱6,299.00",None,/img/jacket.svg,/product/eclipse-shell-jacket.html
5,NR-006,Static Puffer Vest,Outerwear,"₱5,999.00",4.8,/img/vest.svg,/product/static-puffer-vest.html
6,NR-007,Onyx Boxy Tee,Tops,"₱1,599.00",4.7,/img/tee.svg,/product/onyx-boxy-tee.html
7,NR-008,Shadow Leather Gloves,Accessories,"₱1,599.00",4.4,/img/gloves.svg,/product/shadow-leather-gloves.html
8,NR-009,Night Coach Jacket,Outerwear,"₱3,499.00",4.0,/img/jacket.svg,/product/night-coach-jacket.html
9,NR-010,Onyx Mesh Top,Tops,"₱1,599.00",3.9,/img/tee.svg,/product/onyx-mesh-top.html


In [41]:
assert len(products) == 12, f"Expected 12 products on page 1, got {len(products)}"
assert all(p["name"] and p["price"] for p in products), "A product is missing its name or price"
assert len({p["product_id"] for p in products}) == 12, "Duplicate product IDs on page 1"
print("Page 1 OK:", len(products), "products")

Page 1 OK: 12 products


### B-3.4.4

Follow pagination across all pages with a delay and a descriptive user-agent string (Lab 3.4 step 4).

In [42]:
# Returns None if a disallowed URl is visited

def polite_get(url):
    if not robots.can_fetch(USER_AGENT, url):
        print(f"Blocked by robots.txt: {url}")
        return None

    sleep(DELAY)
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    response.encoding = "utf-8"
    return response

In [43]:
url = BASE_URL + "/page-1.html" # Start on Page 1
all_products = []
visited_pages = set()

while url:
    if url in visited_pages:
        raise ValueError(f"Pagination linked back to a page we already visited: {url}")

    response = polite_get(url)
    if response is None:
        break

    soup = BeautifulSoup(response.text, "html.parser")
    all_products.extend(parse_products(soup, url))
    visited_pages.add(url)

    next_link = soup.select_one("a.next")
    url = urljoin(url, next_link["href"]) if next_link else None

unique_ids = {product["product_id"] for product in all_products}

In [44]:
assert len(all_products) == 60, f"Expected 60 products, got {len(all_products)}."
assert len(unique_ids) == 60, f"Expected 60 unique product IDs, got {len(unique_ids)}."

print(f"Collected {len(all_products)} products across {len(visited_pages)} pages.")

Collected 60 products across 5 pages.


Before every request, the scraper checks that `robots.txt` allows the page, waits the crawl delay (1 second), and sends my user agent. It also tracks visited pages, so a "Next" link pointing back to an earlier page raises an error instead of looping forever. The last page has no "Next" link, so the loop ends normally after 5 pages and 60 products.

### B-3.4.5

Add a check that skips disallowed URLs and show it refusing the `/admin/` path (Lab 3.4 step 5).

In [45]:
admin_url = BASE_URL + "/admin/"
admin_response = polite_get(admin_url)

assert admin_response is None, "Expected the scraper to refuse /admin/."

Blocked by robots.txt: http://sandbox-site/admin/


### B-3.4.6

Clean the results (price to float) and save them to `data/products.csv` (Lab 3.4 step 6).

In [46]:
products_df = pd.DataFrame(all_products)

print("Before the cleaning the conversion:")
products_df.head()

Before the cleaning the conversion:


,product_id,name,category,price,rating,image_url,product_url
0,NR-001,Steel Track Jacket,Outerwear,"₱4,199.00",3.4,http://sandbox-site/img/jacket.svg,http://sandbox-site/product/steel-track-jacket...
1,NR-002,Raw Coach Jacket,Outerwear,"₱1,889.30",4.1,http://sandbox-site/img/jacket.svg,http://sandbox-site/product/raw-coach-jacket.html
2,NR-003,Raw Knit Polo,Tops,"₱1,599.00",4.8,http://sandbox-site/img/tee.svg,http://sandbox-site/product/raw-knit-polo.html
3,NR-004,Eclipse Hoodie,Tops,"₱1,699.00",3.8,http://sandbox-site/img/hoodie.svg,http://sandbox-site/product/eclipse-hoodie.html
4,NR-005,Eclipse Shell Jacket,Outerwear,"₱6,299.00",None,http://sandbox-site/img/jacket.svg,http://sandbox-site/product/eclipse-shell-jack...


In [47]:
# Convert the price to float and clean the peso sign and comma ₱4,199.00 → 4199.00
products_df["price"] = products_df["price"].str.replace(r"[₱,]", "", regex=True).astype(float)

# None becomes NaN in the rating field
products_df["rating"] = pd.to_numeric(products_df["rating"])

In [48]:
assert products_df["price"].notna().all() and (products_df["price"] > 0).all(), "Bad price after cleaning"
assert len(products_df) >= 30, "The brief needs at least 30 records"

products_df.to_csv("../data/module3/products.csv", index=False)
print(f"Saved {len(products_df)} products.")

print("Products without a rating:", products_df["rating"].isna().sum())

Saved 60 products.
Products without a rating: 8


`pd.to_numeric` made rating a *float* column, so the missing ratings changed from `None` to `NaN`, the float version of missing.

In [49]:
print(products_df.dtypes)
products_df.head()

product_id      object
name            object
category        object
price          float64
rating         float64
image_url       object
product_url     object
dtype: object


,product_id,name,category,price,rating,image_url,product_url
0,NR-001,Steel Track Jacket,Outerwear,4199.0,3.4,http://sandbox-site/img/jacket.svg,http://sandbox-site/product/steel-track-jacket...
1,NR-002,Raw Coach Jacket,Outerwear,1889.3,4.1,http://sandbox-site/img/jacket.svg,http://sandbox-site/product/raw-coach-jacket.html
2,NR-003,Raw Knit Polo,Tops,1599.0,4.8,http://sandbox-site/img/tee.svg,http://sandbox-site/product/raw-knit-polo.html
3,NR-004,Eclipse Hoodie,Tops,1699.0,3.8,http://sandbox-site/img/hoodie.svg,http://sandbox-site/product/eclipse-hoodie.html
4,NR-005,Eclipse Shell Jacket,Outerwear,6299.0,NaN,http://sandbox-site/img/jacket.svg,http://sandbox-site/product/eclipse-shell-jack...


## Diagrams

### 3-A Timestamp normalization

```mermaid
flowchart LR
    A[Raw timestamps] --> B[Parse per source format]
    B --> C[Count bad rows as NaT]
    B --> D[Localize to Asia/Manila]
    D --> E[Convert to UTC]
    D --> F[Date parts and daily counts]
```

### 3-B Format conversion map

```mermaid
flowchart LR
    J[Nested JSON] --> DF[Flat DataFrame]
    DF --> CSV
    DF --> Excel
    DF --> JSON
    DF --> Avro
    CSV & Excel & Avro --> R[Read back and compare]
```

### 3-C Responsible scraping flow

```mermaid
sequenceDiagram
    participant S as Scraper
    participant W as sandbox-site
    S->>W: GET /robots.txt
    W-->>S: Rules + crawl delay
    loop each listing page
        S->>S: Allowed? Wait 1 s
        S->>W: GET page
        W-->>S: HTML
        S->>S: Parse, follow Next
    end
    S->>S: Skip /admin/, save CSV
```

## What broke and how I fixed it

| # | Lab | What broke | Why | Fix |
|---|---|---|---|---|
| 1 | 3.2 | `FutureWarning: Mismatched null-like values <NA> and None found` when comparing the CSV read-back with the original | The CSV was read with `dtype="string"`, so missing emails became `<NA>`, while the original still had `None` from the JSON | Cast the original to the same dtypes (`.astype(column_types)`) before comparing, so both sides use `<NA>` |
| 2 | 3.4 | Prices came out as `â±4,199.00` and ratings as `â 3.4` | nginx sends `Content-Type: text/html` without a charset, so `requests` decoded the UTF-8 page as ISO-8859-1 | Passed `response.content` (raw bytes) to BeautifulSoup, which reads the page's `<meta charset="utf-8">` |

## Reflection

The hardest part for me was figuring out why Avro rejected a missing email when the schema allowed `null`. It made sense once I saw that pandas had read the empty cell as `NaN`, a float, which I had to convert to `None` before writing to Avro. I was surprised that the peso sign showed up correctly even after I removed the explicit UTF-8 setting; it worked because I passed `response.content` to BeautifulSoup, which reads the page's own `<meta charset="utf-8">`. Next time, I'd check one product card against the page before running the scraper across all five pages, so I can catch things like a garbled price early.